# Data Preparation rõ ràng cho Daily Revenue/COGS Forecasting

Notebook này triển khai lại kế hoạch trong `plan.md` theo kiểu dễ đọc cho intern: mỗi bước lớn có giải thích ngắn, code cell ngắn, output kiểm tra ngay sau khi xử lý, và chỉ thêm biểu đồ khi nó giúp kiểm tra pipeline.

Nguyên tắc chính: dùng EDA ngày 28/5 làm nền, không làm lại EDA; ở đây chỉ chuẩn bị dữ liệu sạch, đúng grain ngày, không leakage, sẵn sàng cho modeling.


## 00. Bản đồ quy trình

**Vì sao cần?** Trước khi code, cần biết notebook đang đi qua những bước nào và mỗi bước trả lời câu hỏi gì.

Flow chuẩn bị dữ liệu:

1. Setup config và helper.
2. Load 14 CSV và kiểm tra dữ liệu vào.
3. Correct type: parse date, convert numeric.
4. Cleaning tối thiểu: duplicate, invalid value, outlier flag.
5. Tạo base daily từ `sales`.
6. Aggregate các nguồn feature về daily grain.
7. Join feature tables và kiểm tra row count.
8. Tạo lag/rolling features để tránh leakage.
9. Xử lý missing cuối và tạo feature schema.
10. Diagnostic target/feature: histogram, scatter, correlation.
11. Kiểm tra outlier bằng context nghiệp vụ: giá bán so với COGS, discount so với gross, refund so với item gốc, target ngày so với volume.
12. Chỉ xử lý/cap outlier nếu có bằng chứng cần thiết.
13. Chia train/valid; kiểm tra provided test từ `sample_submission`.
14. Export artifact và report.


## 01. Setup thư viện và config

**Vì sao cần?** Tập trung toàn bộ đường dẫn, target, tỷ lệ split và danh sách file ở một nơi để dễ sửa. Thư mục output dùng bản mới `outputs/data_preparation_daily_intern` để tránh lẫn file test cũ.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.tseries.holiday import USFederalHolidayCalendar

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs/data_preparation_daily_intern")
CLEAN_TABLE_DIR = OUTPUT_DIR / "clean_tables"
REPORT_DIR = OUTPUT_DIR / "reports"
for folder in [OUTPUT_DIR, CLEAN_TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = ["Revenue", "COGS"]
DATE_KEY = "Date"
FORECAST_LAGS = (1, 7, 14, 28, 365)
FORECAST_WINDOWS = (7, 28, 90)
MIN_LOOKBACK_DAYS = max(FORECAST_LAGS)
VALID_START_DATE = pd.Timestamp("2021-01-01")
ALLOW_SAME_DAY_FEATURES = False

print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


In [ ]:
CORE_FILES = {
    "sales": "sales.csv",
    "sample_submission": "sample_submission.csv",
}

OPTIONAL_FEATURE_FILES = {
    "web_traffic": "web_traffic.csv",
    "customers": "customers.csv",
    "products": "products.csv",
    "returns": "returns.csv",
    "reviews": "reviews.csv",
    "promotions": "promotions.csv",
    "inventory": "inventory.csv",
    "geography": "geography.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "payments": "payments.csv",
    "shipments": "shipments.csv",
}

EXPECTED_FILES = {**CORE_FILES, **OPTIONAL_FEATURE_FILES}

DATE_COLUMNS = {
    "sales": ["Date"],
    "sample_submission": ["Date"],
    "web_traffic": ["date"],
    "orders": ["order_date"],
    "returns": ["return_date"],
    "reviews": ["review_date"],
    "customers": ["signup_date"],
    "promotions": ["start_date", "end_date"],
    "shipments": ["ship_date", "delivery_date"],
    "inventory": ["snapshot_date"],
}

NUMERIC_COLUMNS = {
    "sales": ["Revenue", "COGS"],
    "sample_submission": ["Revenue", "COGS"],
    "web_traffic": ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec"],
    "order_items": ["quantity", "unit_price", "discount_amount"],
    "products": ["price", "cogs"],
    "returns": ["return_quantity", "refund_amount"],
    "reviews": ["rating"],
    "promotions": ["discount_value", "stackable_flag", "min_order_value"],
    "inventory": ["stock_on_hand", "units_received", "units_sold", "stockout_days", "days_of_supply", "fill_rate"],
}

CORE_REQUIRED_COLUMNS = {
    "sales": ["Date", "Revenue", "COGS"],
    "sample_submission": ["Date", "Revenue", "COGS"],
}

OPTIONAL_REQUIRED_COLUMNS = {
    "web_traffic": ["date"],
    "orders": ["order_id", "order_date"],
    "order_items": ["order_id", "product_id", "quantity", "unit_price"],
    "products": ["product_id", "cogs"],
    "returns": ["return_date"],
    "reviews": ["review_date", "rating"],
    "customers": ["customer_id", "signup_date"],
    "promotions": ["start_date", "end_date", "discount_value"],
    "inventory": ["snapshot_date", "product_id"],
}


## 02. Helper nhỏ dùng lại

**Vì sao cần?** Các hàm này chỉ để code bên dưới ngắn hơn. Hàm không tự quyết định nghiệp vụ; mọi thay đổi dữ liệu quan trọng đều ghi vào log.


In [ ]:
cleaning_log_rows = []
warning_rows = []

# Ghi lại mọi hành động làm thay đổi dữ liệu.
def log_action(table, column, action, rows, reason):
    cleaning_log_rows.append({
        "table_name": table,
        "column_name": column,
        "action": action,
        "rows_affected": int(rows),
        "reason": reason,
    })

# Ghi warning để notebook không crash vì thiếu cột phụ.
def warn(issue, step, suggestion=""):
    warning_rows.append({"issue": issue, "step": step, "suggestion": suggestion})

# Kiểm tra nhanh một dataframe có đủ cột cần không.
def has_cols(df, cols):
    return all(c in df.columns for c in cols)


In [ ]:
# Plot đơn giản để kiểm tra pipeline, không dùng để kể lại EDA.
def bar_plot(df, x, y, title, figsize=(8, 4), color="#4C78A8"):
    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(df[y], df[x], color=color)
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

# Tạo lag và rolling từ dữ liệu quá khứ. Rolling luôn bắt đầu từ t-1 để tránh leakage.
def add_lag_and_rolling(df, date_col, source_cols, lags=FORECAST_LAGS, windows=FORECAST_WINDOWS):
    df = df.sort_values(date_col).copy()
    created = []
    for col in source_cols:
        if col not in df.columns:
            continue
        for lag in lags:
            new_col = f"{col}_lag_{lag}d"
            df[new_col] = df[col].shift(lag)
            created.append(new_col)
        rolling_base = df[col].shift(1)
        for window in windows:
            new_col = f"{col}_rolling_{window}d_mean"
            df[new_col] = rolling_base.rolling(window, min_periods=1).mean()
            created.append(new_col)
    return df, created


# IQR helper dùng chung cho flag outlier và kiểm tra ngữ cảnh kinh doanh.
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

def winsorize_series(s, lower_q=0.01, upper_q=0.99):
    lower, upper = s.quantile([lower_q, upper_q])
    return s.clip(lower, upper), lower, upper


## 03. Load raw data

**Vì sao cần?** Data Preparation không được sửa dữ liệu gốc. Ta load vào `raw_tables`, sau đó mới copy sang `tables` để xử lý.


In [ ]:
raw_tables = {}
load_rows = []

for table, file_name in EXPECTED_FILES.items():
    path = DATA_DIR / file_name
    file_role = "core" if table in CORE_FILES else "optional_feature"
    if not path.exists():
        status = "missing_core" if file_role == "core" else "missing_optional"
        warn(f"Missing file: {file_name}", "load", "Core file is required; optional file only removes related features")
        load_rows.append([table, file_name, file_role, status, 0, 0])
        continue

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    raw_tables[table] = df
    load_rows.append([table, file_name, file_role, "loaded", len(df), df.shape[1]])

load_status_report = pd.DataFrame(load_rows, columns=["table_name", "file_name", "file_role", "status", "rows", "columns"])
tables = {name: df.copy() for name, df in raw_tables.items()}

display(load_status_report)


In [ ]:
loaded_for_plot = load_status_report.query("status == 'loaded'").sort_values("rows")
bar_plot(loaded_for_plot, "rows", "table_name", "Kiểm tra nhanh: số dòng theo bảng", figsize=(8, 5))


## 04. Correct type: parse date và convert numeric

**Vì sao cần?** Forecasting theo thời gian bắt buộc date phải đúng kiểu. Các cột tiền/số lượng phải là numeric trước khi aggregate, lag, rolling.


In [ ]:
date_parse_rows = []

for table, cols in DATE_COLUMNS.items():
    if table not in tables:
        continue
    for col in cols:
        if col not in tables[table].columns:
            warn(f"Missing date column: {table}.{col}", "parse_date", "Dependent features will be skipped")
            continue
        original_missing = tables[table][col].isna().sum()
        parsed = pd.to_datetime(tables[table][col], errors="coerce")
        invalid = int(parsed.isna().sum() - original_missing)
        tables[table][col] = parsed.dt.normalize()
        date_parse_rows.append([table, col, round(parsed.notna().mean() * 100, 3), parsed.min(), parsed.max(), invalid])
        if invalid:
            log_action(table, col, "coerce_invalid_date_to_NaT", invalid, "Need valid dates for daily forecasting")

date_parse_report = pd.DataFrame(date_parse_rows, columns=["table_name", "date_column", "parse_success_pct", "min_date", "max_date", "invalid_count"])
display(date_parse_report)


In [ ]:
numeric_conversion_rows = []

for table, cols in NUMERIC_COLUMNS.items():
    if table not in tables:
        continue
    for col in cols:
        if col not in tables[table].columns:
            continue
        before_missing = tables[table][col].isna().sum()
        converted = pd.to_numeric(tables[table][col], errors="coerce")
        invalid = int(converted.isna().sum() - before_missing)
        tables[table][col] = converted
        numeric_conversion_rows.append([table, col, invalid, str(converted.dtype)])
        if invalid:
            log_action(table, col, "coerce_invalid_numeric_to_NaN", invalid, "Need numeric columns for aggregation")

numeric_conversion_report = pd.DataFrame(numeric_conversion_rows, columns=["table_name", "column_name", "invalid_after_conversion", "dtype_after"])
display(numeric_conversion_report)


## 05. Validation tối thiểu trước cleaning

**Vì sao cần?** Đây không phải EDA lại từ đầu. Ta chỉ kiểm tra các cột/key bắt buộc để pipeline không sai grain hoặc join nhầm.


In [ ]:
validation_rows = []

for table, cols in CORE_REQUIRED_COLUMNS.items():
    if table not in tables:
        validation_rows.append(["required_table", table, "", "fail", 1, "Missing core table"])
        continue
    missing = [c for c in cols if c not in tables[table].columns]
    validation_rows.append(["required_columns", table, ", ".join(cols), "pass" if not missing else "fail", len(missing), ", ".join(missing)])

for table, cols in OPTIONAL_REQUIRED_COLUMNS.items():
    if table not in tables:
        validation_rows.append(["optional_table", table, "", "warning", 1, "Optional feature source missing; skip related features"])
        continue
    missing = [c for c in cols if c not in tables[table].columns]
    validation_rows.append(["optional_columns", table, ", ".join(cols), "pass" if not missing else "warning", len(missing), ", ".join(missing)])

for table, date_col in [("sales", "Date"), ("sample_submission", "Date")]:
    if table in tables and date_col in tables[table].columns:
        dup = int(tables[table].duplicated(date_col).sum())
        validation_rows.append(["duplicate_date_key", table, date_col, "pass" if dup == 0 else "warning", dup, "Aggregate/drop duplicate Date"])

minimal_validation_report = pd.DataFrame(validation_rows, columns=["check_name", "table_name", "checked_columns", "status", "issue_count", "note"])
blocking_issue_report = pd.DataFrame(warning_rows)

display(minimal_validation_report)
display(blocking_issue_report)


## 06. Cleaning: duplicate, invalid value, outlier flag

**Vì sao cần?** Cleaning ở đây có ba mức: sửa an toàn, flag để người dùng biết, và không động vào phần chưa có rule. Outlier chỉ flag, không cắt/cap tự động vì EDA đã chỉ ra outlier có thể là đơn lớn hoặc campaign thật.


In [ ]:
clean_tables = {name: df.copy() for name, df in tables.items()}
cleaning_impact_before = {name: (len(df), df.shape[1]) for name, df in clean_tables.items()}

# Drop exact duplicate rows: đây là xử lý an toàn nhất.
for table, df in list(clean_tables.items()):
    dup_rows = int(df.duplicated().sum())
    if dup_rows:
        clean_tables[table] = df.drop_duplicates().copy()
        log_action(table, "*", "drop_exact_duplicate_rows", dup_rows, "Exact duplicate rows add no information")

cleaning_decision_log = pd.DataFrame(cleaning_log_rows)
display(cleaning_decision_log)


In [ ]:
invalid_rows = []

# Flag giá trị âm ở các cột money/count. Không sửa âm thầm.
for table, cols in NUMERIC_COLUMNS.items():
    if table not in clean_tables:
        continue
    for col in cols:
        if col not in clean_tables[table].columns:
            continue
        if col in ["bounce_rate", "fill_rate"]:
            continue
        neg_count = int((clean_tables[table][col] < 0).fillna(False).sum())
        if neg_count:
            invalid_rows.append([table, col, "negative_value", neg_count, "flag_only"])

invalid_value_report = pd.DataFrame(invalid_rows, columns=["table_name", "column_name", "issue", "count", "default_action"])
display(invalid_value_report)


In [ ]:
OUTLIER_CHECK_COLUMNS = {
    "sales": ["Revenue", "COGS"],
    "order_items": ["quantity", "unit_price", "discount_amount"],
    "returns": ["refund_amount"],
    "web_traffic": ["sessions", "page_views"],
}

outlier_rows = []
for table, cols in OUTLIER_CHECK_COLUMNS.items():
    if table not in clean_tables:
        continue
    for col in cols:
        if col not in clean_tables[table].columns:
            continue
        s = clean_tables[table][col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_count = int(((s < lower) | (s > upper)).sum())
        outlier_rows.append([table, col, outlier_count, round(outlier_count / len(s) * 100, 3), lower, upper, "flag_only"])

outlier_report = pd.DataFrame(outlier_rows, columns=["table_name", "column_name", "outlier_count", "outlier_pct", "lower_iqr", "upper_iqr", "action"])
display(outlier_report.sort_values("outlier_count", ascending=False))


In [ ]:
plot_outliers = outlier_report.assign(field=lambda d: d.table_name + "." + d.column_name).sort_values("outlier_count")
bar_plot(plot_outliers, "outlier_count", "field", "Outlier flag theo IQR (chỉ flag, không sửa)", figsize=(8, 4), color="#F58518")


## 07. Tạo daily base từ sales

**Vì sao cần?** Bài toán là dự báo daily `Revenue` và `COGS`, nên bảng gốc để model phải có đúng 1 dòng/ngày từ `sales.Date`.


In [ ]:
sales = clean_tables["sales"][["Date", "Revenue", "COGS"]].dropna(subset=["Date"]).copy()

# Nếu có nhiều row cùng ngày, cộng target theo ngày để giữ đúng grain.
if sales.duplicated("Date").any():
    before = len(sales)
    sales = sales.groupby("Date", as_index=False).agg({"Revenue": "sum", "COGS": "sum"})
    log_action("sales", "Date", "aggregate_duplicate_dates", before - len(sales), "Daily base requires one row per Date")

daily_level_base = sales.sort_values("Date").copy()
daily_level_base["day_of_week"] = daily_level_base["Date"].dt.dayofweek
daily_level_base["month"] = daily_level_base["Date"].dt.month
daily_level_base["quarter"] = daily_level_base["Date"].dt.quarter
daily_level_base["year"] = daily_level_base["Date"].dt.year
daily_level_base["is_weekend"] = daily_level_base["day_of_week"].isin([5, 6]).astype(int)

holidays = USFederalHolidayCalendar().holidays(
    start=daily_level_base["Date"].min(),
    end=daily_level_base["Date"].max(),
).normalize()
daily_level_base["is_holiday"] = daily_level_base["Date"].isin(holidays).astype(int)
daily_level_base["days_to_year_end"] = (
    daily_level_base["Date"].dt.year.map(lambda y: pd.Timestamp(f"{y}-12-31")) - daily_level_base["Date"]
).dt.days

display(daily_level_base.head())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily_level_base["Date"], daily_level_base["Revenue"], label="Revenue", linewidth=1)
ax.plot(daily_level_base["Date"], daily_level_base["COGS"], label="COGS", linewidth=1)
ax.set_title("Kiểm tra target theo thời gian")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
plt.tight_layout()
plt.show()


## 08. Aggregate feature sources về daily grain

**Vì sao cần?** Không được join bảng 1-n trực tiếp vào daily base. Mỗi nguồn phải aggregate về 1 dòng/ngày trước, rồi mới join.


In [ ]:
# Traffic: tổng theo ngày + pivot sessions theo source để giữ tín hiệu từng kênh.
traffic_daily_features = pd.DataFrame(columns=["Date"])
if "web_traffic" in clean_tables and "date" in clean_tables["web_traffic"].columns:
    traffic = clean_tables["web_traffic"].copy()
    traffic_daily_features = traffic.groupby("date", as_index=False).agg(
        sessions_sum=("sessions", "sum"),
        unique_visitors_sum=("unique_visitors", "sum"),
        page_views_sum=("page_views", "sum"),
        avg_bounce_rate=("bounce_rate", "mean"),
        avg_session_duration_sec_mean=("avg_session_duration_sec", "mean"),
        traffic_source_count=("traffic_source", "nunique"),
    ).rename(columns={"date": "Date"})

    if "traffic_source" in traffic.columns:
        sessions_by_source = (
            traffic.pivot_table(index="date", columns="traffic_source", values="sessions", aggfunc="sum", fill_value=0)
            .reset_index()
            .rename(columns={"date": "Date"})
        )
        sessions_by_source.columns = [
            "Date" if c == "Date" else f"sessions_source_{str(c).replace(' ', '_')}"
            for c in sessions_by_source.columns
        ]
        traffic_daily_features = traffic_daily_features.merge(sessions_by_source, on="Date", how="left", validate="one_to_one")

display(traffic_daily_features.head())


In [ ]:
# Orders: gom số đơn, số khách, trạng thái đơn theo ngày đặt hàng.
orders_daily_features = pd.DataFrame(columns=["Date"])
if "orders" in clean_tables and has_cols(clean_tables["orders"], ["order_id", "order_date"]):
    orders = clean_tables["orders"].copy()
    orders_daily_features = orders.groupby("order_date", as_index=False).agg(
        daily_order_count=("order_id", "nunique"),
        daily_customer_count=("customer_id", "nunique"),
        order_source_count=("order_source", "nunique"),
    ).rename(columns={"order_date": "Date"})

    status_counts = pd.crosstab(orders["order_date"], orders["order_status"]).reset_index().rename(columns={"order_date": "Date"})
    status_counts = status_counts.rename(columns={c: f"order_status_{c}_count" for c in status_counts.columns if c != "Date"})
    orders_daily_features = orders_daily_features.merge(status_counts, on="Date", how="left")

display(orders_daily_features.head())


In [ ]:
# Items: join order_items với orders/products trước, rồi aggregate theo order_date.
items_daily_features = pd.DataFrame(columns=["Date"])
needed = "order_items" in clean_tables and "orders" in clean_tables
if needed and has_cols(clean_tables["order_items"], ["order_id", "product_id", "quantity", "unit_price"]):
    items = clean_tables["order_items"].merge(
        clean_tables["orders"][["order_id", "order_date"]],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
    if "products" in clean_tables and has_cols(clean_tables["products"], ["product_id", "cogs"]):
        items = items.merge(clean_tables["products"][["product_id", "cogs"]], on="product_id", how="left", validate="many_to_one")

    items["gross_item_revenue"] = items["quantity"] * items["unit_price"]
    items["estimated_cogs"] = items["quantity"] * items.get("cogs", 0)

    items_daily_features = items.groupby("order_date", as_index=False).agg(
        daily_total_quantity=("quantity", "sum"),
        daily_gross_item_revenue=("gross_item_revenue", "sum"),
        daily_discount_amount=("discount_amount", "sum"),
        daily_estimated_cogs=("estimated_cogs", "sum"),
        daily_distinct_products=("product_id", "nunique"),
    ).rename(columns={"order_date": "Date"})

display(items_daily_features.head())


In [ ]:
# Returns: return là future/post-event nên chỉ dùng dạng lag ở bước sau.
returns_daily_features = pd.DataFrame(columns=["Date"])
if "returns" in clean_tables and "return_date" in clean_tables["returns"].columns:
    returns_daily_features = clean_tables["returns"].groupby("return_date", as_index=False).agg(
        daily_return_count=("return_id", "count"),
        daily_return_quantity=("return_quantity", "sum"),
        daily_refund_amount=("refund_amount", "sum"),
    ).rename(columns={"return_date": "Date"})

display(returns_daily_features.head())


In [ ]:
# Inventory: snapshot theo sản phẩm/tháng -> aggregate monthly rồi forward-fill về daily trước khi join.
inventory_daily_features = pd.DataFrame(columns=["Date"])
if "inventory" in clean_tables and "snapshot_date" in clean_tables["inventory"].columns:
    inventory_snapshot_features = clean_tables["inventory"].groupby("snapshot_date", as_index=False).agg(
        daily_stock_on_hand_sum=("stock_on_hand", "sum"),
        daily_units_received_sum=("units_received", "sum"),
        daily_units_sold_sum=("units_sold", "sum"),
        daily_stockout_days_sum=("stockout_days", "sum"),
        daily_days_of_supply_mean=("days_of_supply", "mean"),
        daily_fill_rate_mean=("fill_rate", "mean"),
        daily_inventory_product_count=("product_id", "nunique"),
    ).rename(columns={"snapshot_date": "Date"}).sort_values("Date")

    date_index = pd.date_range(daily_level_base["Date"].min(), daily_level_base["Date"].max(), freq="D")
    inventory_daily_features = (
        inventory_snapshot_features
        .set_index("Date")
        .reindex(date_index)
        .ffill()
        .reset_index()
        .rename(columns={"index": "Date"})
    )

display(inventory_daily_features.head())


In [ ]:
# Promotions/reviews/customers: tín hiệu bổ sung, aggregate về một dòng/ngày.
promo_daily_features = pd.DataFrame(columns=["Date"])
if "promotions" in clean_tables and has_cols(clean_tables["promotions"], ["start_date", "end_date", "discount_value"]):
    promo = clean_tables["promotions"].dropna(subset=["start_date", "end_date"]).copy()
    promo_daily_features = pd.DataFrame({"Date": pd.date_range(daily_level_base["Date"].min(), daily_level_base["Date"].max(), freq="D")})

    def active_promo_stats(date):
        active = promo[(promo["start_date"] <= date) & (promo["end_date"] >= date)]
        return pd.Series({
            "promo_count_active": len(active),
            "promo_avg_discount": active["discount_value"].mean() if len(active) else 0,
            "promo_max_discount": active["discount_value"].max() if len(active) else 0,
        })

    promo_daily_features = pd.concat([promo_daily_features, promo_daily_features["Date"].apply(active_promo_stats)], axis=1)
    promo_daily_features["is_promo_active"] = (promo_daily_features["promo_count_active"] > 0).astype(int)

reviews_daily_features = pd.DataFrame(columns=["Date"])
if "reviews" in clean_tables and has_cols(clean_tables["reviews"], ["review_date", "rating"]):
    reviews_daily_features = clean_tables["reviews"].groupby("review_date", as_index=False).agg(
        daily_review_count=("rating", "count"),
        daily_avg_rating=("rating", "mean"),
        daily_low_rating_count=("rating", lambda x: (x <= 2).sum()),
    ).rename(columns={"review_date": "Date"})

customers_daily_features = pd.DataFrame(columns=["Date"])
if "customers" in clean_tables and has_cols(clean_tables["customers"], ["customer_id", "signup_date"]):
    customers_daily_features = clean_tables["customers"].groupby("signup_date", as_index=False).agg(
        daily_new_customers=("customer_id", "nunique"),
    ).rename(columns={"signup_date": "Date"})

display(promo_daily_features.head())
display(reviews_daily_features.head())
display(customers_daily_features.head())


In [ ]:
daily_feature_tables = {
    "traffic_daily_features": traffic_daily_features,
    "orders_daily_features": orders_daily_features,
    "items_daily_features": items_daily_features,
    "returns_daily_features": returns_daily_features,
    "inventory_daily_features": inventory_daily_features,
    "promo_daily_features": promo_daily_features,
    "reviews_daily_features": reviews_daily_features,
    "customers_daily_features": customers_daily_features,
}

aggregate_rows = []
for name, df in daily_feature_tables.items():
    aggregate_rows.append([name, len(df), df["Date"].nunique() if "Date" in df.columns else 0, int(df.duplicated("Date").sum()) if "Date" in df.columns else 0])

aggregation_quality_report = pd.DataFrame(aggregate_rows, columns=["output_table", "rows", "unique_dates", "duplicate_dates"])
display(aggregation_quality_report)


## 09. Join feature tables vào daily base

**Vì sao cần?** Sau join, số dòng phải giữ nguyên bằng `daily_level_base`. Nếu số dòng tăng, tức là có row explosion.


In [ ]:
daily_model_base = daily_level_base.copy()
join_rows = []

for name, feature_df in daily_feature_tables.items():
    if len(feature_df) == 0 or "Date" not in feature_df.columns:
        continue
    before = len(daily_model_base)
    feature_cols = [c for c in feature_df.columns if c != "Date"]
    daily_model_base = daily_model_base.merge(feature_df, on="Date", how="left", validate="one_to_one")
    after = len(daily_model_base)
    unmatched = int(daily_model_base[feature_cols].isna().all(axis=1).sum()) if feature_cols else 0
    join_rows.append([name, before, after, unmatched, "pass" if before == after else "fail"])

join_quality_report = pd.DataFrame(join_rows, columns=["join_step", "rows_before", "rows_after", "unmatched_left_count", "status"])
display(join_quality_report)


In [ ]:
coverage_plot = join_quality_report.assign(matched_rows=lambda d: d.rows_after - d.unmatched_left_count)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(coverage_plot["join_step"], coverage_plot["matched_rows"], label="matched")
ax.bar(coverage_plot["join_step"], coverage_plot["unmatched_left_count"], bottom=coverage_plot["matched_rows"], label="unmatched")
ax.set_title("Kiểm tra coverage sau join")
ax.set_ylabel("rows")
ax.tick_params(axis="x", rotation=35)
ax.legend()
plt.tight_layout()
plt.show()


## 10. Leakage-safe feature engineering

**Vì sao cần?** Daily forecast không được dùng thông tin cùng ngày hoặc tương lai nếu tại thời điểm dự báo chưa biết. Vì vậy target và nguồn event đều được chuyển thành lag/rolling features.


In [ ]:
promo_cols = [c for c in ["promo_count_active", "promo_avg_discount", "promo_max_discount", "is_promo_active"] if c in daily_model_base.columns]
calendar_cols = ["day_of_week", "month", "quarter", "year", "is_weekend", "is_holiday", "days_to_year_end"] + promo_cols

daily_model_base, target_lag_cols = add_lag_and_rolling(
    daily_model_base, "Date", TARGET_COLUMNS, lags=FORECAST_LAGS, windows=FORECAST_WINDOWS
)

same_day_source_cols = [
    c for c in daily_model_base.columns
    if c not in ["Date"] + TARGET_COLUMNS + calendar_cols + target_lag_cols
    and pd.api.types.is_numeric_dtype(daily_model_base[c])
]

daily_model_base, external_lag_cols = add_lag_and_rolling(
    daily_model_base, "Date", same_day_source_cols, lags=FORECAST_LAGS, windows=FORECAST_WINDOWS
)

print("Known calendar/promo features:", len(calendar_cols))
print("Target lag features:", len(target_lag_cols))
print("External lag features:", len(external_lag_cols))
print("Same-day source columns excluded by default:", len(same_day_source_cols))


In [ ]:
leakage_rows = []
for col in TARGET_COLUMNS:
    leakage_rows.append([col, "sales", "target", "exclude_from_X", "Target column"])
for col in calendar_cols:
    leakage_rows.append([col, "calendar", "low", "include", "Known before prediction"])
for col in target_lag_cols + external_lag_cols:
    leakage_rows.append([col, "lagged_or_rolling", "low", "include", "Uses only shifted historical values"])
for col in same_day_source_cols:
    leakage_rows.append([col, "same_day_realized", "medium/high", "exclude_by_default", "May not be known at prediction time"])

leakage_audit_table = pd.DataFrame(leakage_rows, columns=["feature_name", "source_group", "leakage_risk", "default_action", "reason"])
display(leakage_audit_table.head(30))


## 11. Final missing handling và feature schema

**Vì sao cần?** Model không nhận missing trong X. Nhưng target thì không được fill. Ta chỉ fill feature, và ghi rõ phương pháp trong report.


In [ ]:
included_features = calendar_cols + target_lag_cols + external_lag_cols
included_features = [c for c in included_features if c in daily_model_base.columns]

feature_table_daily = daily_model_base[["Date"] + TARGET_COLUMNS + included_features].copy()
missing_before = feature_table_daily[included_features].isna().sum()


In [ ]:
imputation_rows = []
for col in included_features:
    before = int(feature_table_daily[col].isna().sum())
    if before == 0:
        imputation_rows.append([col, before, "none", "No missing", "not_needed"])
        continue

    if any(token in col for token in ["count", "quantity", "sessions", "visitors", "views", "stock", "return", "refund", "discount", "promo"]):
        method = "fill_0"
        reason = "Missing means no available event/history"
    else:
        method = "train_median"
        reason = "Fit median on train only, then apply to valid"

    imputation_rows.append([col, before, method, reason, "fit_after_time_split"])

final_imputation_report = pd.DataFrame(
    imputation_rows,
    columns=["feature_name", "missing_before", "method", "reason", "fit_stage"],
)
display(final_imputation_report.sort_values("missing_before", ascending=False).head(20))


In [ ]:
missing_plot = final_imputation_report.query("missing_before > 0").sort_values("missing_before").tail(20)
if len(missing_plot):
    bar_plot(missing_plot, "missing_before", "feature_name", "Top missing trước imputation", figsize=(8, 6), color="#E45756")


In [ ]:
final_feature_schema = pd.DataFrame({
    "feature_name": ["Date"] + TARGET_COLUMNS + included_features + same_day_source_cols,
})
final_feature_schema["included_in_X"] = final_feature_schema["feature_name"].isin(included_features)
final_feature_schema["feature_group"] = np.select(
    [
        final_feature_schema.feature_name.eq("Date"),
        final_feature_schema.feature_name.isin(TARGET_COLUMNS),
        final_feature_schema.feature_name.isin(calendar_cols),
        final_feature_schema.feature_name.isin(same_day_source_cols),
    ],
    ["id", "target", "calendar", "excluded_same_day"],
    default="lagged_numeric",
)
final_feature_schema["reason"] = np.where(final_feature_schema.included_in_X, "Model feature", "Key/target/leakage exclusion")

target_definition_table = pd.DataFrame([
    [target, "sales", "daily", int(feature_table_daily[target].isna().sum()), "ready"]
    for target in TARGET_COLUMNS
], columns=["target_column", "source_table", "grain", "missing_count", "status"])

display(final_feature_schema.head(30))
display(target_definition_table)


## 12. Feature diagnostics: target distribution, linearity, correlation

**Vì sao cần?** Sau khi có feature sạch, cần nhìn nhanh target và feature trước khi modeling. Phần này không thay thế EDA, nhưng giúp trả lời: target có lệch không, feature nào liên quan mạnh với `Revenue`/`COGS`, quan hệ có gần tuyến tính không, và có feature nào quá trùng nhau không.


In [ ]:
# Bảng coverage: notebook đã làm kỹ thuật clean/feature engineering nào.
clean_feature_coverage = pd.DataFrame([
    ["clean", "load raw tables", "done", "raw_tables + tables"],
    ["clean", "strip column names", "done", "df.columns strip whitespace"],
    ["clean", "correct date type", "done", "date_parse_report"],
    ["clean", "correct numeric type", "done", "numeric_conversion_report"],
    ["clean", "exact duplicate handling", "done", "drop exact duplicates if any"],
    ["clean", "required column validation", "done", "minimal_validation_report"],
    ["clean", "invalid negative value check", "done", "invalid_value_report"],
    ["clean", "outlier detection", "done_flag_only", "IQR flag, no automatic capping"],
    ["clean", "missing imputation in X", "done_train_only", "fit values on train, transform valid"],
    ["clean", "business context outlier validation", "done", "business_outlier_rulebook + context reports"],
    ["clean", "target imputation", "not_done_by_design", "Do not fill Revenue/COGS"],
    ["feature_engineering", "daily aggregation", "done", "traffic/orders/items/returns/inventory/promotions/reviews/customers"],
    ["feature_engineering", "web traffic source pivot", "done", "sessions_source_* features"],
    ["feature_engineering", "calendar features", "done", "weekend/holiday/days_to_year_end/promo"],
    ["feature_engineering", "lag features", "done", "1d, 7d, 14d, 28d, 365d"],
    ["feature_engineering", "rolling features", "done", "7d, 28d, 90d mean"],
    ["feature_engineering", "same-day leakage exclusion", "done", "same_day_source_cols excluded"],
    ["feature_engineering", "categorical encoding", "not_needed_v1", "Current X is numeric/calendar only"],
    ["feature_engineering", "scaling", "not_done_yet", "Usually belongs to modeling pipeline"],
    ["feature_engineering", "feature-target correlation", "done", "feature_target_correlation_report"],
    ["feature_engineering", "linearity scatter check", "done", "scatter plots for top correlated features"],
], columns=["stage", "technique", "status", "evidence"])

display(clean_feature_coverage)


In [ ]:
# Histogram target để kiểm tra phân phối và độ lệch.
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

axes[0, 0].hist(feature_table_daily["Revenue"], bins=50, color="#4C78A8")
axes[0, 0].set_title("Revenue distribution")
axes[0, 1].hist(np.log1p(feature_table_daily["Revenue"].clip(lower=0)), bins=50, color="#72B7B2")
axes[0, 1].set_title("log1p(Revenue) distribution")

axes[1, 0].hist(feature_table_daily["COGS"], bins=50, color="#F58518")
axes[1, 0].set_title("COGS distribution")
axes[1, 1].hist(np.log1p(feature_table_daily["COGS"].clip(lower=0)), bins=50, color="#E45756")
axes[1, 1].set_title("log1p(COGS) distribution")

for ax in axes.ravel():
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()


In [ ]:
# Tính Pearson/Spearman giữa từng feature và target.
feature_target_rows = []
for target in TARGET_COLUMNS:
    for feature in included_features:
        pair = feature_table_daily[[feature, target]].dropna()
        if len(pair) < 30 or pair[feature].nunique() <= 1:
            continue
        pearson = pair[feature].corr(pair[target], method="pearson")
        spearman = pair[feature].corr(pair[target], method="spearman")
        feature_target_rows.append([feature, target, pearson, spearman, abs(pearson), abs(spearman)])

feature_target_correlation_report = pd.DataFrame(
    feature_target_rows,
    columns=["feature_name", "target", "pearson_corr", "spearman_corr", "abs_pearson", "abs_spearman"],
).sort_values(["target", "abs_pearson"], ascending=[True, False])

display(feature_target_correlation_report.groupby("target").head(15))


In [ ]:
# Scatter top features để nhìn tuyến tính với Revenue/COGS.
# Nếu scatter tạo thành dải gần đường thẳng thì mô hình tuyến tính có thể hợp lý hơn.
# Nếu cong, cụm, hoặc có outlier mạnh thì có thể cần tree model, transform, hoặc robust loss.
for target in TARGET_COLUMNS:
    top_features = (
        feature_target_correlation_report
        .query("target == @target")
        .dropna(subset=["pearson_corr"])
        .head(4)["feature_name"]
        .tolist()
    )
    if not top_features:
        continue

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes = axes.ravel()
    for ax, feature in zip(axes, top_features):
        sample = feature_table_daily[[feature, target]].dropna()
        if len(sample) > 1500:
            sample = sample.sample(1500, random_state=42)
        ax.scatter(sample[feature], sample[target], s=8, alpha=0.35)
        corr = sample[feature].corr(sample[target])
        ax.set_title(f"{feature} | Pearson={corr:.3f}")
        ax.set_xlabel(feature)
        ax.set_ylabel(target)
    for ax in axes[len(top_features):]:
        ax.axis("off")
    fig.suptitle(f"Scatter kiểm tra tuyến tính với {target}", y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# Correlation heatmap nhỏ cho các feature liên quan mạnh nhất.
# Mục tiêu: phát hiện feature quá trùng nhau/multicollinearity trước modeling.
top_corr_features = (
    feature_target_correlation_report
    .sort_values("abs_pearson", ascending=False)
    .drop_duplicates("feature_name")
    .head(12)["feature_name"]
    .tolist()
)
heatmap_cols = TARGET_COLUMNS + top_corr_features
corr_matrix = feature_table_daily[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_xticklabels(corr_matrix.columns, rotation=75, ha="right")
ax.set_yticklabels(corr_matrix.index)
ax.set_title("Correlation heatmap: target và top features")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# Báo cáo feature có tương quan rất cao với nhau để modeling lưu ý.
high_corr_pairs = []
feature_corr = feature_table_daily[top_corr_features].corr().abs()
for i, left in enumerate(feature_corr.columns):
    for right in feature_corr.columns[i + 1:]:
        corr_value = feature_corr.loc[left, right]
        if corr_value >= 0.95:
            high_corr_pairs.append([left, right, corr_value])

high_feature_correlation_report = pd.DataFrame(high_corr_pairs, columns=["feature_a", "feature_b", "abs_corr"])
display(high_feature_correlation_report)


## 13. Business context checks cho outlier

**Vì sao cần?** Một giá trị cao chưa chắc là lỗi. Ví dụ `unit_price` cao có thể hợp lệ nếu `cogs` cũng cao và margin không vô lý. Phần này đối chiếu outlier với các cột nghiệp vụ liên quan để phân loại: `likely_valid`, `review`, hoặc `possible_error`.


In [ ]:
# Rulebook: outlier chỉ bị coi là lỗi khi mâu thuẫn với context nghiệp vụ.
business_outlier_rulebook = pd.DataFrame([
    ["unit_price", "unit_price cao + cogs cao + margin hợp lý", "likely_valid", "Sản phẩm premium/giá trị cao có thể hợp lệ"],
    ["unit_price", "unit_price thấp hơn cogs đáng kể", "possible_error", "Bán dưới giá vốn bất thường, cần kiểm tra price/promo/input"],
    ["unit_price", "unit_price/cogs quá lớn", "review", "Có thể markup cao hoặc lỗi giá, cần xem product/category"],
    ["discount", "discount <= gross line value và rate trong vùng campaign", "likely_valid", "Khuyến mãi lớn nhưng logic tài chính vẫn đúng"],
    ["discount", "discount > gross line value", "possible_error", "Chiết khấu vượt giá trị dòng hàng"],
    ["refund", "refund gần bằng gross item gốc", "likely_valid", "Hoàn tiền toàn phần/gần toàn phần"],
    ["refund", "refund > gross item gốc", "possible_error", "Hoàn tiền vượt giá trị mua"],
    ["daily_target", "Revenue/COGS cao đi cùng orders/quantity/cogs cao", "likely_valid", "Ngày bán lớn thật"],
    ["daily_target", "target cao nhưng margin âm hoặc thiếu context", "review", "Cần xem thêm discount/refund/join"],
], columns=["check_area", "rule", "validity_group", "reason"])

display(business_outlier_rulebook)


In [ ]:
# Chuẩn bị bảng item-level có đủ order date, product price/cogs để kiểm tra outlier theo nghiệp vụ.
item_context = pd.DataFrame()
if "order_items" in clean_tables and "orders" in clean_tables:
    item_context = clean_tables["order_items"].merge(
        clean_tables["orders"][["order_id", "order_date"]],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
    if "products" in clean_tables and has_cols(clean_tables["products"], ["product_id", "price", "cogs", "category"]):
        item_context = item_context.merge(
            clean_tables["products"][["product_id", "product_name", "category", "price", "cogs"]],
            on="product_id",
            how="left",
            validate="many_to_one",
        )

    item_context["gross_line_value"] = item_context["quantity"] * item_context["unit_price"]
    item_context["discount_rate"] = item_context["discount_amount"] / item_context["gross_line_value"].replace(0, np.nan)
    item_context["unit_margin"] = item_context["unit_price"] - item_context.get("cogs", np.nan)
    item_context["unit_margin_rate"] = item_context["unit_margin"] / item_context["unit_price"].replace(0, np.nan)
    item_context["unit_price_to_cogs_ratio"] = item_context["unit_price"] / item_context.get("cogs", np.nan).replace(0, np.nan)

display(item_context.head())


In [ ]:
# Check 1: unit_price cao có hợp lệ không?
# Logic: giá cao + COGS cao + margin không âm/quá cực đoan => có thể là sản phẩm premium hợp lệ.
if len(item_context):
    _, unit_price_upper = iqr_bounds(item_context["unit_price"].dropna())
    unit_price_outliers = item_context[item_context["unit_price"] > unit_price_upper].copy()

    conditions = [
        unit_price_outliers["cogs"].isna(),
        unit_price_outliers["unit_margin_rate"] < -0.05,
        unit_price_outliers["unit_price_to_cogs_ratio"] > 5,
        unit_price_outliers["cogs"] >= item_context["cogs"].quantile(0.90),
    ]
    choices = [
        "review_missing_cogs",
        "possible_error_price_below_cogs",
        "review_extreme_price_to_cogs_ratio",
        "likely_valid_premium_product_high_cogs",
    ]
    unit_price_outliers["business_validity"] = np.select(conditions, choices, default="review_high_price_context")
    unit_price_outliers["check_reason"] = np.select(
        conditions,
        [
            "Không có COGS để đối chiếu",
            "Unit price thấp hơn COGS đáng kể",
            "Giá bán cao hơn COGS quá nhiều, cần kiểm tra pricing/promo",
            "Giá cao đi cùng COGS cao, có thể là sản phẩm premium hợp lệ",
        ],
        default="Giá cao nhưng cần xem category/product/margin",
    )
    item_price_outlier_context_report = unit_price_outliers[[
        "order_id", "order_date", "product_id", "product_name", "category", "quantity",
        "unit_price", "price", "cogs", "unit_margin", "unit_margin_rate",
        "unit_price_to_cogs_ratio", "business_validity", "check_reason",
    ]].sort_values("unit_price", ascending=False).head(50)
else:
    item_price_outlier_context_report = pd.DataFrame()

display(item_price_outlier_context_report)


In [ ]:
# Check 2: discount outlier có lỗi logic không?
# Logic: discount cao có thể là campaign, nhưng discount > gross line value là lỗi/cần review mạnh.
if len(item_context):
    _, discount_upper = iqr_bounds(item_context["discount_amount"].dropna())
    discount_outliers = item_context[item_context["discount_amount"] > discount_upper].copy()
    discount_outliers["business_validity"] = np.select(
        [
            discount_outliers["discount_amount"] > discount_outliers["gross_line_value"],
            discount_outliers["discount_rate"] > 0.80,
            discount_outliers["discount_rate"].between(0.20, 0.80, inclusive="both"),
        ],
        [
            "possible_error_discount_exceeds_gross",
            "review_extreme_discount_rate",
            "likely_valid_campaign_discount",
        ],
        default="review_discount_context",
    )
    discount_outliers["check_reason"] = np.select(
        [
            discount_outliers["discount_amount"] > discount_outliers["gross_line_value"],
            discount_outliers["discount_rate"] > 0.80,
            discount_outliers["discount_rate"].between(0.20, 0.80, inclusive="both"),
        ],
        [
            "Discount lớn hơn gross line value",
            "Discount rate quá cao, cần kiểm tra campaign/nhập liệu",
            "Discount cao nhưng vẫn nằm trong gross line value, có thể là campaign",
        ],
        default="Discount cao cần xem promo/category/order context",
    )
    discount_outlier_context_report = discount_outliers[[
        "order_id", "order_date", "product_id", "product_name", "category", "quantity",
        "unit_price", "gross_line_value", "discount_amount", "discount_rate",
        "business_validity", "check_reason",
    ]].sort_values("discount_amount", ascending=False).head(50)
else:
    discount_outlier_context_report = pd.DataFrame()

display(discount_outlier_context_report)


In [ ]:
# Check 3: refund outlier có hợp lệ không?
# Logic: refund cao hợp lệ nếu order/product gốc cũng có gross value cao. Refund > purchased gross là dấu hiệu cần kiểm tra.
if "returns" in clean_tables and len(item_context):
    item_ref_context = (
        item_context
        .groupby(["order_id", "product_id"], as_index=False)
        .agg(
            order_date=("order_date", "min"),
            product_name=("product_name", "first"),
            category=("category", "first"),
            quantity=("quantity", "sum"),
            unit_price=("unit_price", "mean"),
            gross_line_value=("gross_line_value", "sum"),
            cogs=("cogs", "mean"),
        )
    )
    return_context = clean_tables["returns"].merge(
        item_ref_context,
        on=["order_id", "product_id"],
        how="left",
        validate="many_to_one",
    )
    return_context["refund_to_gross_ratio"] = return_context["refund_amount"] / return_context["gross_line_value"].replace(0, np.nan)
    _, refund_upper = iqr_bounds(return_context["refund_amount"].dropna())
    refund_outliers = return_context[return_context["refund_amount"] > refund_upper].copy()
    refund_outliers["business_validity"] = np.select(
        [
            refund_outliers["gross_line_value"].isna(),
            refund_outliers["refund_to_gross_ratio"] > 1.05,
            refund_outliers["refund_to_gross_ratio"].between(0.70, 1.05, inclusive="both"),
            refund_outliers["gross_line_value"] >= return_context["gross_line_value"].quantile(0.90),
        ],
        [
            "review_missing_original_item",
            "possible_error_refund_exceeds_gross",
            "likely_valid_full_or_near_full_refund",
            "likely_valid_high_value_item_refund",
        ],
        default="review_refund_context",
    )
    refund_outliers["check_reason"] = np.select(
        [
            refund_outliers["gross_line_value"].isna(),
            refund_outliers["refund_to_gross_ratio"] > 1.05,
            refund_outliers["refund_to_gross_ratio"].between(0.70, 1.05, inclusive="both"),
            refund_outliers["gross_line_value"] >= return_context["gross_line_value"].quantile(0.90),
        ],
        [
            "Không map được item gốc để đối chiếu",
            "Refund lớn hơn gross value của item gốc",
            "Refund gần bằng giá trị mua, có thể là hoàn tiền đầy đủ",
            "Refund cao vì item/order gốc cũng có giá trị cao",
        ],
        default="Refund cao cần xem reason/quantity/gross value",
    )
    refund_outlier_context_report = refund_outliers[[
        "return_id", "order_id", "product_id", "return_date", "return_reason", "return_quantity",
        "refund_amount", "gross_line_value", "refund_to_gross_ratio", "product_name", "category",
        "business_validity", "check_reason",
    ]].sort_values("refund_amount", ascending=False).head(50)
else:
    refund_outlier_context_report = pd.DataFrame()

display(refund_outlier_context_report)


In [ ]:
# Check 4: daily Revenue/COGS outlier có hợp lệ không?
# Logic: ngày doanh thu cao hợp lệ nếu đi cùng order_count/quantity/COGS cao và margin không bất thường.
daily_context = daily_level_base.merge(
    orders_daily_features[["Date", "daily_order_count", "daily_customer_count"]],
    on="Date", how="left"
).merge(
    items_daily_features[["Date", "daily_total_quantity", "daily_gross_item_revenue", "daily_estimated_cogs", "daily_distinct_products"]],
    on="Date", how="left"
)
daily_context["gross_margin"] = daily_context["Revenue"] - daily_context["COGS"]
daily_context["gross_margin_rate"] = daily_context["gross_margin"] / daily_context["Revenue"].replace(0, np.nan)
daily_context["revenue_per_order"] = daily_context["Revenue"] / daily_context["daily_order_count"].replace(0, np.nan)

daily_target_context_rows = []
for target in TARGET_COLUMNS:
    _, upper = iqr_bounds(daily_context[target].dropna())
    temp = daily_context[daily_context[target] > upper].copy()
    temp["target"] = target
    temp["target_value"] = temp[target]
    temp["business_validity"] = np.select(
        [
            temp["gross_margin_rate"] < -0.05,
            temp["daily_order_count"] >= daily_context["daily_order_count"].quantile(0.90),
            temp["daily_total_quantity"] >= daily_context["daily_total_quantity"].quantile(0.90),
            temp["daily_estimated_cogs"] >= daily_context["daily_estimated_cogs"].quantile(0.90),
        ],
        [
            "review_negative_margin_day",
            "likely_valid_high_volume_day",
            "likely_valid_high_quantity_day",
            "likely_valid_high_cost_high_sales_day",
        ],
        default="review_target_outlier_context",
    )
    temp["check_reason"] = np.select(
        [
            temp["gross_margin_rate"] < -0.05,
            temp["daily_order_count"] >= daily_context["daily_order_count"].quantile(0.90),
            temp["daily_total_quantity"] >= daily_context["daily_total_quantity"].quantile(0.90),
            temp["daily_estimated_cogs"] >= daily_context["daily_estimated_cogs"].quantile(0.90),
        ],
        [
            "Target cao nhưng margin âm, cần kiểm tra COGS/discount/refund",
            "Target cao đi cùng số đơn cao",
            "Target cao đi cùng số lượng bán cao",
            "Target cao đi cùng estimated COGS cao",
        ],
        default="Target cao nhưng chưa đủ context để kết luận hợp lệ",
    )
    daily_target_context_rows.append(temp[[
        "Date", "target", "target_value", "daily_order_count", "daily_customer_count", "daily_total_quantity",
        "daily_gross_item_revenue", "daily_estimated_cogs", "gross_margin_rate", "revenue_per_order",
        "business_validity", "check_reason",
    ]])

daily_target_outlier_context_report = pd.concat(daily_target_context_rows, ignore_index=True) if daily_target_context_rows else pd.DataFrame()
display(daily_target_outlier_context_report.sort_values("target_value", ascending=False).head(50))


In [ ]:
# Tóm tắt phân loại context để nhìn nhanh outlier nào hợp lệ, outlier nào cần review.
context_reports = {
    "unit_price": item_price_outlier_context_report,
    "discount": discount_outlier_context_report,
    "refund": refund_outlier_context_report,
    "daily_target": daily_target_outlier_context_report,
}

business_outlier_validity_summary = []
for name, df in context_reports.items():
    if len(df) == 0 or "business_validity" not in df.columns:
        continue
    counts = df["business_validity"].value_counts().reset_index()
    counts.columns = ["business_validity", "rows"]
    counts.insert(0, "context_check", name)
    business_outlier_validity_summary.append(counts)

business_outlier_validity_summary = pd.concat(business_outlier_validity_summary, ignore_index=True) if business_outlier_validity_summary else pd.DataFrame()
display(business_outlier_validity_summary)


## 14. Outlier treatment decision support

**Vì sao cần?** Outlier không nên xóa/cap tự động. Ta cần nhìn bảng và biểu đồ để quyết định. Phần này tạo một bản winsorize thử nghiệm để so sánh tác động, nhưng **mặc định không áp dụng vào final data**. Nếu sau khi xem evidence thấy cần xử lý cho mô hình tuyến tính, bật `APPLY_OUTLIER_TREATMENT_TO_X = True`.


In [ ]:
# Config xử lý outlier: analysis-only mặc định.
APPLY_OUTLIER_TREATMENT_TO_X = False
WINSOR_LOWER_Q = 0.01
WINSOR_UPPER_Q = 0.99

outlier_treatment_config = pd.DataFrame([
    ["APPLY_OUTLIER_TREATMENT_TO_X", APPLY_OUTLIER_TREATMENT_TO_X, "False = chỉ phân tích, không thay đổi X"],
    ["WINSOR_LOWER_Q", WINSOR_LOWER_Q, "Ngưỡng dưới cho winsorize thử nghiệm"],
    ["WINSOR_UPPER_Q", WINSOR_UPPER_Q, "Ngưỡng trên cho winsorize thử nghiệm"],
    ["target_policy", "do_not_cap_target_by_default", "Target nên giữ raw; cân nhắc log-transform ở modeling"],
    ["feature_policy", "cap_only_if_needed_for_linear_model", "Tree/boosting thường chịu outlier tốt hơn linear model"],
], columns=["config", "value", "note"])

display(outlier_treatment_config)


In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

def winsorize_series(s, lower_q=WINSOR_LOWER_Q, upper_q=WINSOR_UPPER_Q):
    lower, upper = s.quantile([lower_q, upper_q])
    return s.clip(lower, upper), lower, upper

# Chọn target + top feature tương quan cao để soi outlier kỹ hơn.
top_corr_features_for_outlier = (
    feature_target_correlation_report
    .sort_values("abs_pearson", ascending=False)
    .drop_duplicates("feature_name")
    .head(12)["feature_name"]
    .tolist()
)
outlier_decision_columns = TARGET_COLUMNS + [c for c in top_corr_features_for_outlier if c not in TARGET_COLUMNS]

outlier_treatment_rows = []
for col in outlier_decision_columns:
    s = feature_table_daily[col].dropna()
    if len(s) == 0:
        continue
    lower_iqr, upper_iqr = iqr_bounds(s)
    outlier_mask = (s < lower_iqr) | (s > upper_iqr)
    wins, lower_cap, upper_cap = winsorize_series(s)
    outlier_pct = outlier_mask.mean() * 100
    skew_value = s.skew()

    if col in TARGET_COLUMNS:
        recommendation = "keep_raw_target; consider log1p in modeling"
    elif outlier_pct >= 10:
        recommendation = "review; winsorize if using linear model"
    elif abs(skew_value) >= 2:
        recommendation = "consider log/winsorize for linear model"
    else:
        recommendation = "keep_by_default"

    outlier_treatment_rows.append({
        "column_name": col,
        "role": "target" if col in TARGET_COLUMNS else "feature",
        "rows": len(s),
        "iqr_outlier_count": int(outlier_mask.sum()),
        "iqr_outlier_pct": round(outlier_pct, 3),
        "skew": round(skew_value, 3),
        "p01_cap": lower_cap,
        "p99_cap": upper_cap,
        "mean_before": s.mean(),
        "mean_after_winsor_trial": wins.mean(),
        "std_before": s.std(),
        "std_after_winsor_trial": wins.std(),
        "recommendation": recommendation,
    })

outlier_treatment_recommendation = pd.DataFrame(outlier_treatment_rows)
display(outlier_treatment_recommendation.sort_values(["role", "iqr_outlier_pct"], ascending=[True, False]))


In [ ]:
# Ví dụ những dòng target outlier lớn nhất để xem có phải ngày bán thật hay lỗi.
target_outlier_examples = []
for target in TARGET_COLUMNS:
    s = feature_table_daily[target]
    lower_iqr, upper_iqr = iqr_bounds(s.dropna())
    extreme = feature_table_daily.loc[s > upper_iqr, ["Date", target]].copy()
    extreme["target"] = target
    extreme["iqr_upper_bound"] = upper_iqr
    extreme["distance_above_bound"] = extreme[target] - upper_iqr
    extreme = extreme.sort_values(target, ascending=False).head(10)
    target_outlier_examples.append(extreme.rename(columns={target: "value"}))

target_outlier_examples = pd.concat(target_outlier_examples, ignore_index=True) if target_outlier_examples else pd.DataFrame()
display(target_outlier_examples)


In [ ]:
# Biểu đồ before/after winsorize thử nghiệm cho top outlier columns.
plot_cols = (
    outlier_treatment_recommendation
    .query("iqr_outlier_count > 0")
    .sort_values("iqr_outlier_pct", ascending=False)
    .head(6)["column_name"]
    .tolist()
)

if plot_cols:
    ncols = 2
    nrows = int(np.ceil(len(plot_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, plot_cols):
        raw = feature_table_daily[col].dropna()
        wins, _, _ = winsorize_series(raw)
        ax.hist(raw, bins=50, alpha=0.45, label="raw")
        ax.hist(wins, bins=50, alpha=0.45, label="winsor trial")
        ax.set_title(col)
        ax.legend()
    for ax in axes[len(plot_cols):]:
        ax.axis("off")
    fig.suptitle("Before/after winsorize thử nghiệm", y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# So sánh correlation với target trước/sau winsorize thử nghiệm.
outlier_impact_rows = []
for col in outlier_decision_columns:
    raw = feature_table_daily[col].dropna()
    wins, lower_cap, upper_cap = winsorize_series(raw)
    capped_count = int((raw.ne(wins)).sum())

    row = {
        "column_name": col,
        "role": "target" if col in TARGET_COLUMNS else "feature",
        "capped_rows_trial": capped_count,
        "capped_pct_trial": round(capped_count / len(raw) * 100, 3) if len(raw) else 0,
        "lower_cap": lower_cap,
        "upper_cap": upper_cap,
        "mean_before": raw.mean(),
        "mean_after": wins.mean(),
        "std_before": raw.std(),
        "std_after": wins.std(),
    }

    # Với feature, xem correlation với Revenue/COGS thay đổi thế nào nếu cap thử nghiệm.
    for target in TARGET_COLUMNS:
        if col == target:
            row[f"corr_with_{target}_before"] = np.nan
            row[f"corr_with_{target}_after"] = np.nan
            continue
        aligned = feature_table_daily[[col, target]].dropna().copy()
        if len(aligned) < 30 or aligned[col].nunique() <= 1:
            row[f"corr_with_{target}_before"] = np.nan
            row[f"corr_with_{target}_after"] = np.nan
            continue
        aligned_wins = aligned[col].clip(lower_cap, upper_cap)
        row[f"corr_with_{target}_before"] = aligned[col].corr(aligned[target])
        row[f"corr_with_{target}_after"] = aligned_wins.corr(aligned[target])

    outlier_impact_rows.append(row)

outlier_treatment_impact_report = pd.DataFrame(outlier_impact_rows)
display(outlier_treatment_impact_report.sort_values("capped_pct_trial", ascending=False))


In [ ]:
# Tạo bảng model input. Mặc định dùng raw feature_table_daily; nếu bật config thì winsorize feature trong X.
feature_table_model_input = feature_table_daily.copy()

if APPLY_OUTLIER_TREATMENT_TO_X:
    feature_cols_to_cap = outlier_treatment_recommendation.query("role == 'feature'")["column_name"].tolist()
    for col in feature_cols_to_cap:
        if col in feature_table_model_input.columns:
            feature_table_model_input[col], _, _ = winsorize_series(feature_table_model_input[col])
    print("Applied winsorize treatment to selected X features.")
else:
    print("Outlier treatment is analysis-only. Final X keeps raw features.")


## 15. Chia train/valid và kiểm tra provided test

**Vì sao cần?** Không chia test từ lịch sử nữa vì đã có provided test trong `sample_submission`. Ta chỉ chia lịch sử thành train/valid theo thời gian, rồi kiểm tra provided test có giống schema target/date và không overlap không.


In [ ]:
model_rows = feature_table_model_input.sort_values("Date").dropna(subset=TARGET_COLUMNS).copy()
model_rows = model_rows[model_rows["Date"] >= model_rows["Date"].min() + pd.Timedelta(days=MIN_LOOKBACK_DAYS)]

train_df = model_rows[model_rows["Date"] < VALID_START_DATE].copy()
valid_df = model_rows[model_rows["Date"] >= VALID_START_DATE].copy()

if len(train_df) == 0 or len(valid_df) == 0:
    fallback_start = model_rows["Date"].quantile(0.70)
    train_df = model_rows[model_rows["Date"] < fallback_start].copy()
    valid_df = model_rows[model_rows["Date"] >= fallback_start].copy()

impute_detail_rows = []
method_lookup = final_imputation_report.set_index("feature_name")["method"].to_dict()
for col in included_features:
    train_before = int(train_df[col].isna().sum())
    valid_before = int(valid_df[col].isna().sum())
    method = method_lookup.get(col, "none")

    if method in ["none", "fill_0"]:
        fill_value = 0 if method == "fill_0" else np.nan
    else:
        fill_value = train_df[col].median()
        if pd.isna(fill_value):
            fill_value = 0

    if method != "none":
        train_df[col] = train_df[col].fillna(fill_value)
        valid_df[col] = valid_df[col].fillna(fill_value)

    impute_detail_rows.append([
        col, method, fill_value,
        train_before, int(train_df[col].isna().sum()),
        valid_before, int(valid_df[col].isna().sum()),
    ])

impute_detail_report = pd.DataFrame(
    impute_detail_rows,
    columns=["feature_name", "method", "fill_value_from_train", "train_missing_before", "train_missing_after", "valid_missing_before", "valid_missing_after"],
)
final_imputation_report = final_imputation_report.merge(impute_detail_report, on=["feature_name", "method"], how="left")

X_train = train_df[included_features].copy()
X_valid = valid_df[included_features].copy()
y_train_revenue = train_df[["Date", "Revenue"]].copy()
y_valid_revenue = valid_df[["Date", "Revenue"]].copy()
y_train_cogs = train_df[["Date", "COGS"]].copy()
y_valid_cogs = valid_df[["Date", "COGS"]].copy()

def split_summary_row(name, df):
    return [name, len(df), df["Date"].min(), df["Date"].max(), df["Revenue"].mean(), df["COGS"].mean()]

split_summary = pd.DataFrame(
    [split_summary_row("train", train_df), split_summary_row("valid", valid_df)],
    columns=["split", "rows", "date_min", "date_max", "revenue_mean", "cogs_mean"],
)
display(split_summary)
display(final_imputation_report.sort_values(["train_missing_before", "valid_missing_before"], ascending=False).head(20))


In [ ]:
provided_test_table = clean_tables["sample_submission"].dropna(subset=["Date"]).drop_duplicates("Date").sort_values("Date").copy()

provided_test_schema_report = pd.DataFrame([
    ["has_Date", "pass" if "Date" in provided_test_table.columns else "fail"],
    ["has_Revenue", "pass" if "Revenue" in provided_test_table.columns else "warning"],
    ["has_COGS", "pass" if "COGS" in provided_test_table.columns else "warning"],
    ["unique_Date", "pass" if provided_test_table["Date"].is_unique else "fail"],
    ["no_train_valid_overlap", "pass" if provided_test_table["Date"].isin(model_rows["Date"]).sum() == 0 else "warning"],
    ["target_numeric", "pass" if all(pd.api.types.is_numeric_dtype(provided_test_table[c]) for c in TARGET_COLUMNS) else "warning"],
], columns=["check_name", "status"])

provided_test_summary = pd.DataFrame([{
    "rows": len(provided_test_table),
    "date_min": provided_test_table["Date"].min(),
    "date_max": provided_test_table["Date"].max(),
    "overlaps_train_valid_dates": int(provided_test_table["Date"].isin(model_rows["Date"]).sum()),
    "has_revenue": "Revenue" in provided_test_table.columns,
    "has_cogs": "COGS" in provided_test_table.columns,
}])

display(provided_test_summary)
display(provided_test_schema_report)


In [ ]:
def target_stats(name, df):
    return {
        "dataset": name,
        "rows": len(df),
        "revenue_mean": df["Revenue"].mean(),
        "revenue_std": df["Revenue"].std(),
        "cogs_mean": df["COGS"].mean(),
        "cogs_std": df["COGS"].std(),
        "date_min": df["Date"].min(),
        "date_max": df["Date"].max(),
    }

target_distribution_compare = pd.DataFrame([
    target_stats("train", train_df),
    target_stats("valid", valid_df),
    target_stats("provided_test", provided_test_table),
])
display(target_distribution_compare)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(target_distribution_compare))
ax.bar(x - 0.18, target_distribution_compare["revenue_mean"], width=0.36, label="Revenue mean")
ax.bar(x + 0.18, target_distribution_compare["cogs_mean"], width=0.36, label="COGS mean")
ax.set_xticks(x)
ax.set_xticklabels(target_distribution_compare["dataset"])
ax.set_title("So sánh target mean: train / valid / provided test")
ax.legend()
plt.tight_layout()
plt.show()


## 16. Export artifact và report

**Vì sao cần?** Modeling nên dùng file đã chuẩn bị sẵn, không phải chạy lại toàn bộ cleaning mỗi lần. Export manifest giúp biết file nào dùng cho việc gì.


In [ ]:
export_objects = {
    "daily_model_base.csv": daily_model_base,
    "feature_table_daily.csv": feature_table_daily,
    "feature_table_model_input.csv": feature_table_model_input,
    "X_train.csv": X_train,
    "X_valid.csv": X_valid,
    "y_train_revenue.csv": y_train_revenue,
    "y_valid_revenue.csv": y_valid_revenue,
    "y_train_cogs.csv": y_train_cogs,
    "y_valid_cogs.csv": y_valid_cogs,
    "provided_test_table.csv": provided_test_table,
    "final_feature_schema.csv": final_feature_schema,
}

manifest_rows = []
for file_name, df in export_objects.items():
    path = OUTPUT_DIR / file_name
    df.to_csv(path, index=False)
    manifest_rows.append([file_name, "csv", len(df), df.shape[1], "modeling artifact", str(path)])

for name, df in clean_tables.items():
    path = CLEAN_TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    manifest_rows.append([f"clean_tables/{name}.csv", "csv", len(df), df.shape[1], "clean source table", str(path)])


In [ ]:
report_tables = {
    "business_outlier_rulebook": business_outlier_rulebook,
    "load_status_report": load_status_report,
    "date_parse_report": date_parse_report,
    "numeric_conversion_report": numeric_conversion_report,
    "minimal_validation_report": minimal_validation_report,
    "cleaning_decision_log": cleaning_decision_log,
    "invalid_value_report": invalid_value_report,
    "outlier_report": outlier_report,
    "item_price_outlier_context_report": item_price_outlier_context_report,
    "discount_outlier_context_report": discount_outlier_context_report,
    "refund_outlier_context_report": refund_outlier_context_report,
    "daily_target_outlier_context_report": daily_target_outlier_context_report,
    "business_outlier_validity_summary": business_outlier_validity_summary,
    "outlier_treatment_config": outlier_treatment_config,
    "outlier_treatment_recommendation": outlier_treatment_recommendation,
    "target_outlier_examples": target_outlier_examples,
    "outlier_treatment_impact_report": outlier_treatment_impact_report,
    "aggregation_quality_report": aggregation_quality_report,
    "join_quality_report": join_quality_report,
    "leakage_audit_table": leakage_audit_table,
    "final_imputation_report": final_imputation_report,
    "final_feature_schema": final_feature_schema,
    "target_definition_table": target_definition_table,
    "split_summary": split_summary,
    "provided_test_summary": provided_test_summary,
    "provided_test_schema_report": provided_test_schema_report,
    "target_distribution_compare": target_distribution_compare,
    "clean_feature_coverage": clean_feature_coverage,
    "feature_target_correlation_report": feature_target_correlation_report,
    "high_feature_correlation_report": high_feature_correlation_report,
}

for report_name, df in report_tables.items():
    path = REPORT_DIR / f"{report_name}.csv"
    df.to_csv(path, index=False)
    manifest_rows.append([f"reports/{report_name}.csv", "csv", len(df), df.shape[1], "audit report", str(path)])

export_manifest = pd.DataFrame(manifest_rows, columns=["file_name", "file_type", "rows", "columns", "purpose", "path"])
export_manifest.to_csv(OUTPUT_DIR / "export_manifest.csv", index=False)
display(export_manifest)


## 17. Final checklist

**Vì sao cần?** Đây là checklist ngắn để biết data đã sẵn sàng sang modeling chưa.


In [ ]:
final_preparation_checklist = pd.DataFrame([
    ["raw data loaded", "pass" if len(raw_tables) else "fail", "load_status_report"],
    ["required columns checked", "pass" if not minimal_validation_report.query("status == 'fail'").shape[0] else "fail", "minimal_validation_report"],
    ["type corrected", "pass", "date_parse_report + numeric_conversion_report"],
    ["duplicates handled", "pass", "cleaning_decision_log"],
    ["outliers flagged", "pass", "outlier_report"],
    ["outliers checked with business context", "pass" if len(business_outlier_validity_summary) else "warn", "business_outlier_validity_summary"],
    ["daily grain valid", "pass" if daily_model_base["Date"].is_unique else "fail", "aggregation_quality_report"],
    ["joins validated", "pass" if not join_quality_report.query("status == 'fail'").shape[0] else "fail", "join_quality_report"],
    ["leakage checked", "pass", "leakage_audit_table"],
    ["features have no missing", "pass" if X_train.isna().sum().sum() + X_valid.isna().sum().sum() == 0 else "fail", "final_imputation_report"],
    ["train/valid split created", "pass" if len(train_df) and len(valid_df) else "fail", "split_summary"],
    ["provided test checked", "pass" if not provided_test_schema_report.query("status == 'fail'").shape[0] else "fail", "provided_test_schema_report"],
    ["outputs exported", "pass" if len(export_manifest) else "fail", "export_manifest"],
], columns=["checklist_item", "status", "evidence_table"])

display(final_preparation_checklist)

print("Final daily forecasting dataset")
print(f"- rows history: {len(feature_table_daily):,}")
print(f"- date range history: {feature_table_daily['Date'].min()} -> {feature_table_daily['Date'].max()}")
print(f"- targets: {', '.join(TARGET_COLUMNS)}")
print(f"- features in X: {len(included_features)}")
print(f"- same-day columns excluded: {len(same_day_source_cols)}")
print(f"- train/valid rows: {len(train_df):,}/{len(valid_df):,}")
print(f"- provided test rows: {len(provided_test_table):,}")
print(f"- output dir: {OUTPUT_DIR}")
print(f"- ready_for_modeling: {'yes' if final_preparation_checklist.status.eq('fail').sum() == 0 else 'no'}")
